# Recommendation Systems Algorithms Project
## 00. Environment & Dataset Accessibility Verification

### A. Project Information
- **Project Title:** Recommendation Systems Algorithms (Öneri Algoritmaları Projesi)
- **Notebook Purpose:** Verify environment setup, package versions, configuration integrity, and raw dataset accessibility.
- **Dataset Path:** Configured in `config.yaml` (external location `C:\ml-32m`)
- **Kernel:** `Recommendation Systems - Project` (`recommendation-systems-project`)


In [1]:
import sys
import os
import platform
import yaml
import numpy as np
import pandas as pd
import scipy
import matplotlib
import seaborn as sns
import sklearn
import ipykernel
import jupyter_core

print('=== Project Information ===')
project_root = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
print('Project Root:', project_root)
print('Working Directory:', os.getcwd())
print('Platform:', platform.platform())
print('Python Executable:', sys.executable)
print('Python Version:', sys.version)


=== Project Information ===
Project Root: C:\OneriAlgoritmalariProjesi
Working Directory: C:\OneriAlgoritmalariProjesi\notebooks
Platform: Windows-11-10.0.26200-SP0
Python Executable: C:\OneriAlgoritmalariProjesi\.venv\Scripts\python.exe
Python Version: 3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]


### B. Environment Verification
Verifying that:
1. Python is executing from the project virtual environment (`.venv`).
2. All required core packages are installed and reporting valid versions.


In [2]:
print('=== Python Environment & Dependencies ===')

is_venv = '.venv' in sys.executable.lower() or hasattr(sys, 'real_prefix') or (hasattr(sys, 'base_prefix') and sys.base_prefix != sys.prefix)
print('Running inside Virtual Environment (.venv):', is_venv, f'({sys.executable})')

packages = {
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scipy': scipy.__version__,
    'matplotlib': matplotlib.__version__,
    'seaborn': sns.__version__,
    'scikit-learn': sklearn.__version__,
    'ipykernel': ipykernel.__version__,
    'pyyaml': yaml.__version__,
    'jupyter_core': jupyter_core.__version__,
}

for pkg, ver in packages.items():
    print(f'  - {pkg:<15}: {ver}')

env_check_passed = is_venv and all(ver is not None for ver in packages.values())
status_label = 'PASS' if env_check_passed else 'FAIL'
print(f'\n[Environment Verification Result]: {status_label}')


=== Python Environment & Dependencies ===
Running inside Virtual Environment (.venv): True (C:\OneriAlgoritmalariProjesi\.venv\Scripts\python.exe)
  - numpy          : 2.5.4
  - pandas         : 3.1.0
  - scipy          : 1.18.1
  - matplotlib     : 3.11.2
  - seaborn        : 0.13.2
  - scikit-learn   : 1.9.1
  - ipykernel      : 7.4.0
  - pyyaml         : 6.0.3
  - jupyter_core   : 5.9.1

[Environment Verification Result]: PASS


### C. Configuration Verification
Reading `config.yaml` from the project root and verifying that all dataset paths and configuration keys exist and resolve properly.


In [3]:
print('=== Configuration Verification ===')

config_path = 'config.yaml' if os.path.isfile('config.yaml') else os.path.join('..', 'config.yaml')
config_path = os.path.abspath(config_path)

print('Configuration file path:', config_path)
assert os.path.isfile(config_path), f'config.yaml not found at {config_path}'

with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print('Parsed configuration content:')
print(yaml.dump(config, default_flow_style=False))

dataset_dir = os.path.abspath(config['dataset']['directory'])
print('Resolved dataset directory:', dataset_dir)
assert os.path.isdir(dataset_dir), f'Dataset directory {dataset_dir} does not exist!'

config_check_passed = os.path.isdir(dataset_dir) and 'random_seed' in config.get('project', {})
status_label = 'PASS' if config_check_passed else 'FAIL'
print(f'[Configuration Verification Result]: {status_label}')


=== Configuration Verification ===
Configuration file path: C:\OneriAlgoritmalariProjesi\config.yaml
Parsed configuration content:
dataset:
  directory: C:/ml-32m
  links: links.csv
  movies: movies.csv
  ratings: ratings.csv
  tags: tags.csv
project:
  random_seed: 42

Resolved dataset directory: C:\ml-32m
[Configuration Verification Result]: PASS


### D. Dataset Accessibility Verification
Verifying that all 4 dataset CSV files (`ratings.csv`, `movies.csv`, `tags.csv`, `links.csv`) exist at the configured external path and can be opened in read-only mode without loading large files into memory.


In [4]:
print('=== Dataset Accessibility Verification ===')

dataset_files = {
    'ratings.csv': config['dataset']['ratings'],
    'movies.csv': config['dataset']['movies'],
    'tags.csv': config['dataset']['tags'],
    'links.csv': config['dataset']['links'],
}

dataset_access_results = {}
for key, filename in dataset_files.items():
    file_path = os.path.join(dataset_dir, filename)
    exists = os.path.isfile(file_path)
    if exists:
        size_bytes = os.path.getsize(file_path)
        size_mb = size_bytes / (1024 * 1024)
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                first_line = f.readline().strip()
            readable = True
        except Exception as e:
            readable = False
            first_line = str(e)
        dataset_access_results[filename] = {
            'exists': exists,
            'size_bytes': size_bytes,
            'size_mb': round(size_mb, 2),
            'readable': readable,
            'header': first_line,
        }
        print(f'File: {filename:<12} | Size: {size_mb:8.2f} MiB ({size_bytes:,} bytes) | Readable: {readable} | Header: {first_line}')
    else:
        dataset_access_results[filename] = {'exists': False, 'readable': False}
        print(f'File: {filename:<12} | MISSING!')

dataset_check_passed = all(info['exists'] and info['readable'] for info in dataset_access_results.values())
status_label = 'PASS' if dataset_check_passed else 'FAIL'
print(f'\n[Dataset Accessibility Result]: {status_label}')


=== Dataset Accessibility Verification ===
File: ratings.csv  | Size:   836.45 MiB (877,076,222 bytes) | Readable: True | Header: userId,movieId,rating,timestamp
File: movies.csv   | Size:     4.05 MiB (4,242,926 bytes) | Readable: True | Header: movieId,title,genres
File: tags.csv     | Size:    69.00 MiB (72,353,890 bytes) | Readable: True | Header: userId,movieId,tag,timestamp
File: links.csv    | Size:     1.86 MiB (1,950,748 bytes) | Readable: True | Header: movieId,imdbId,tmdbId

[Dataset Accessibility Result]: PASS


### E. Final Status & Summary
Consolidated status reporting of all environment and accessibility checks.


In [5]:
print('================ FINAL STATUS REPORT ================')
checks = {
    'Python Virtual Environment (.venv)': env_check_passed,
    'Required Package Imports & Versions': all(ver is not None for ver in packages.values()),
    'Configuration Loading (config.yaml)': config_check_passed,
    'Dataset Directory Resolution': os.path.isdir(dataset_dir),
    'Dataset Files Existence & Readability': dataset_check_passed,
}

all_passed = True
for check_name, status in checks.items():
    status_str = 'PASS' if status else 'FAIL'
    if not status:
        all_passed = False
    print(f' - {check_name:<42}: {status_str}')

print('-----------------------------------------------------')
overall_status = 'PASS' if all_passed else 'FAIL'
print(f' OVERALL VERIFICATION STATUS: {overall_status}')
print('=====================================================')


================ FINAL STATUS REPORT ================
 - Python Virtual Environment (.venv)        : PASS
 - Required Package Imports & Versions       : PASS
 - Configuration Loading (config.yaml)       : PASS
 - Dataset Directory Resolution              : PASS
 - Dataset Files Existence & Readability     : PASS
-----------------------------------------------------
 OVERALL VERIFICATION STATUS: PASS
